# RAGAS 기본 실습 (OpenAI / Gemini 스위칭)

- 이 노트북은 ragas로 RAG 결과를 평가하는 기본 예제를 포함합니다.
- 평가에 사용할 LLM/임베딩을 **OpenAI ↔ Gemini** 사이에서 노트북 설정만 바꿔서 스위칭할 수 있게 구성했습니다.
- 필요 환경 변수
  - OpenAI: `OPENAI_API_KEY`
  - Gemini: `GOOGLE_API_KEY` (Google AI Studio 키)


In [1]:
# ragas 및 필요한 패키지 설치 (최초 1회 실행)
# - ragas 본체
# - HuggingFace datasets
# - Gemini 연동용 langchain-google-genai (3.x 이상)
# - google-generativeai (langchain-google-genai 3.x와 호환되는 버전)
%pip install "ragas[openai]" datasets "langchain-google-genai>=3.0.0" "google-generativeai>=0.8.3"


  Using cached google_generativeai-0.8.5-py3-none-any.whl.metadata (3.9 kB)
INFO: pip is looking at multiple versions of google-generativeai to determine which version is compatible with other requirements. This could take a while.
  Using cached google_generativeai-0.8.4-py3-none-any.whl.metadata (4.2 kB)
  Using cached google_generativeai-0.8.3-py3-none-any.whl.metadata (3.9 kB)
  Using cached langchain_google_genai-3.1.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-3.0.3-py3-none-any.whl.metadata (2.7 kB)
INFO: pip is still looking at multiple versions of google-generativeai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_google_genai-3.0.2-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_google_genai-3.0.1-py3-none-any.whl.metadata (7.1 kB)
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See

In [2]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import AnswerRelevancy, Faithfulness, ContextPrecision
from dotenv import load_dotenv

# .env 또는 환경변수에서 OPENAI_API_KEY 등을 로드
load_dotenv()

data = {
    "question": [
        "What is LangChain?",
        "What is RAG?",
    ],
    "answer": [
        "LangChain is a framework for developing LLM applications.",
        "RAG is retrieval-augmented generation.",
    ],
    "contexts": [
        ["LangChain is a framework for building applications with LLMs."],
        ["Retrieval-augmented generation (RAG) combines retrieval and generation."],
    ],
    "ground_truth": [
        "LangChain is a framework for building LLM-powered applications.",
        "RAG is a technique that combines information retrieval with text generation.",
    ],
}

dataset = Dataset.from_dict(data)

metrics = [
    AnswerRelevancy(),
    Faithfulness(),
    ContextPrecision(),
]

dataset


/Users/dhkim/.pyenv/versions/3.11.14/envs/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 2
})

In [3]:
# 평가에 사용할 LLM / 임베딩 설정 (OpenAI / Gemini 스위칭)
import os
from ragas.llms import llm_factory, LangchainLLMWrapper
from ragas.embeddings import OpenAIEmbeddings, LangchainEmbeddingsWrapper

# "openai" 또는 "gemini" 로 설정해서 스위칭
EVALUATOR_PROVIDER = "gemini"  # 또는 "openai"

# OpenAI 모델 설정
OPENAI_LLM_MODEL = "gpt-4o-mini"
OPENAI_EMBED_MODEL = "text-embedding-3-small"

# Gemini 모델 설정 (Google AI Studio)
GEMINI_LLM_MODEL = "gemini-2.5-flash"
GEMINI_EMBED_MODEL = "models/embedding-001"  # 또는 "models/text-embedding-005"

if EVALUATOR_PROVIDER == "openai":
    # ragas 기본 OpenAI 팩토리 + OpenAIEmbeddings 사용
    evaluator_llm = llm_factory(OPENAI_LLM_MODEL)
    evaluator_embeddings = OpenAIEmbeddings(
        model=OPENAI_EMBED_MODEL,
    )

elif EVALUATOR_PROVIDER == "gemini":
    # Gemini 사용 시 GOOGLE_API_KEY 필수
    if not os.getenv("GOOGLE_API_KEY"):
        raise RuntimeError("Gemini를 사용하려면 GOOGLE_API_KEY 환경 변수가 필요합니다.")

    # 필요할 때만 Gemini용 패키지 import
    from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

    # LangChain LLM/임베딩을 ragas용 래퍼로 감싸기
    evaluator_llm = LangchainLLMWrapper(
        ChatGoogleGenerativeAI(
            model=GEMINI_LLM_MODEL,
            temperature=0.0,
        )
    )
    evaluator_embeddings = LangchainEmbeddingsWrapper(
        GoogleGenerativeAIEmbeddings(
            model=GEMINI_EMBED_MODEL,
            task_type="retrieval_document",
        )
    )

else:
    raise ValueError(f"지원하지 않는 EVALUATOR_PROVIDER 값: {EVALUATOR_PROVIDER}")


/var/folders/tp/hz_s3_r55093y3g2b7b72kfw0000gn/T/ipykernel_75294/3180880607.py:33: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(
/var/folders/tp/hz_s3_r55093y3g2b7b72kfw0000gn/T/ipykernel_75294/3180880607.py:39: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(


In [4]:
result = evaluate(
    dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)
result


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[3]: GoogleGenerativeAIError(Error embedding content: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 0
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 0
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 0
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 0 [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-l

{'answer_relevancy': nan, 'faithfulness': 1.0000, 'context_precision': 1.0000}

In [5]:
result_df = result.to_pandas()
result_df


,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness,context_precision
0,What is LangChain?,[LangChain is a framework for building applica...,LangChain is a framework for developing LLM ap...,LangChain is a framework for building LLM-powe...,NaN,1.0,1.0
1,What is RAG?,[Retrieval-augmented generation (RAG) combines...,RAG is retrieval-augmented generation.,RAG is a technique that combines information r...,NaN,1.0,1.0
